<h1 align="center">
<h1 align="center"> Tarea 2 - Web Structure Mining
<h2 align="center">  Integrantes: Nombre 1, Nombre 2 y Nombre 3.
<h3 align="center"> IN5529 - Inteligencia Artificial Aplicada a la Web
<h3 align="center"> Primavera 2025

---

# Introducción

Ingrese aquí su introducción.

#Instalación e importación de librerias

In [1]:
# Instalaciones
!pip install selenium
!apt-get update
!apt install chromium-chromedriver
import sys
sys.path.insert(0,'/usr/lib/chromium-browser/chromedriver')
!python -m spacy download es_core_news_md
!pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.7/512.7 kB 39.0 MB/s eta 0:00:00
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,080 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,354 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [

In [2]:
# Web Scraper
from selenium import webdriver
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException
from selenium.webdriver.common.by import By

# Librerías varias
import pandas as pd
import numpy as np
import json
import time
import random
import urllib

# Preprocesamiento
import re
import unicodedata
import nltk
nltk.download('punkt')
nltk.download('stopwords')
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import strip_accents_ascii
import spacy
import string

# Ranking
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
import networkx as nx

import nltk

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


# Obtención de datos

## Web Scraper

In [ ]:
# Configuración del Webdriver
from selenium.webdriver.chrome.service import Service
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--lang=es-ES")
chrome_options.add_argument("--user-agent=Mozilla/5.0")

prefs = {
    "profile.managed_default_content_settings.images": 2,
    "profile.managed_default_content_settings.stylesheets": 2,
    "profile.managed_default_content_settings.cookies": 1
}
chrome_options.add_experimental_option("prefs", prefs)

caps = chrome_options.to_capabilities()
caps["pageLoadStrategy"] = "eager"

service = Service()


In [ ]:
#Get links
# Función get_links
# Scrapea todos los links dentro de una página (con timeouts y skip si se cuelga)
from urllib.parse import urlparse, urljoin, urlunparse
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By

def get_links(url, url_base, url_data):
    global chrome_options, service
    print(f"Scrapeando links desde {url}")

    driver = webdriver.Chrome(service=service, options=chrome_options)
    # timeouts cortos para evitar cuelgues
    try:
        driver.set_page_load_timeout(20)   # corta cargas lentas
        driver.set_script_timeout(6)       # corta JS lento
    except Exception:
        pass

    try:
        try:
            driver.get(url)
        except TimeoutException:
            # si tarda cargando, saltamos la página completa
            print("\nget_links: timeout en carga → página saltada")
            return

        # Extrae TODOS los href del cuerpo del artículo de una sola vez (sin .text)
        try:
            js = """
            const scope = document.querySelector('div.mw-parser-output')
                        || document.querySelector('#mw-content-text')
                        || document.body;
            return Array.from(scope.querySelectorAll('a[href]'), a => a.getAttribute('href'));
            """
            hrefs = driver.execute_script(js) or []
        except WebDriverException:
            print("\nget_links: timeout en script → página saltada")
            return

        total = len(hrefs)
        i = 0
        seen = set(url_data)

        for h in hrefs:
            i += 1
            print(f"\r\tLinks revisados: {i}/{total}", end="")
            if not h:
                continue

            # Absolutiza + limpia query/fragment + quita barra final
            abs_url = urljoin(url_base, h)
            p = urlparse(abs_url)._replace(query="", fragment="")
            link = urlunparse(p).rstrip("/")

            # Filtros básicos (ajusta el prefijo si tu wiki no usa /es/wiki/)
            if (url_base in link
                and "mediawiki" not in link
                and "/Archivo:" not in link
                and "/Especial:" not in link
                and "/Ayuda:" not in link
                and "/Categoría:" not in link
                and link not in seen):
                url_data.append(link)
                seen.add(link)

        print(f"\r\tLinks revisados: {total}/{total}")

    except Exception as e:
        print("\nget_links error:", e)
        return
    finally:
        try:
            driver.quit()
        except:
            pass


In [ ]:
def get_text(url):
    d = webdriver.Chrome(service=service, options=chrome_options)
    try:
        # timeouts cortos para evitar bloqueos
        try:
            d.set_page_load_timeout(25)
            d.set_script_timeout(8)
        except Exception:
            pass

        try:
            d.get(url)
        except TimeoutException:
            # si tarda demasiado cargando, seguimos con lo que haya
            pass

        # 1) Intento: mw-parser-output
        try:
            js1 = "var el=document.querySelector('div.mw-parser-output'); return el?el.innerText:'';"
            txt = d.execute_script(js1) or ""
            txt = (txt or "").strip()
            if txt:
                return [txt]
        except WebDriverException:
            pass

        # 2) Fallback: #mw-content-text
        try:
            js2 = "var el=document.querySelector('#mw-content-text'); return el?el.innerText:'';"
            txt2 = d.execute_script(js2) or ""
            txt2 = (txt2 or "").strip()
            if txt2:
                return [txt2]
        except WebDriverException:
            pass

        # 3) Último recurso: body
        try:
            js3 = "return (document.body && document.body.innerText) ? document.body.innerText : '';"
            body_txt = d.execute_script(js3) or ""
            return [ (body_txt or '').strip() ]
        except WebDriverException:
            return [""]

    except Exception as e:
        print("get_text error:", e)
        return [""]
    finally:
        try:
            d.quit()
        except:
            pass

In [ ]:
# Scrapeamos las URLs desde la página principal
url_base = 'https://terraria.fandom.com/es'  # terraria :)
url_data = [url_base]  # Lista de se almacenan las URLs encontradas
n_urls = 100  # Cantidad de URLs que queremos recolectar

for url in url_data:
    # Si todavía no llegamos al límite de URLs deseadas extraemos más enlaces
    if len(url_data) < n_urls:
        get_links(url, url_base, url_data)
    else:
        break

url_data = random.sample(url_data, min(n_urls, len(url_data)))
print('Cantidad de urls:', len(url_data))


Scrapeando links desde https://terraria.fandom.com/es
	Links revisados: 109/109
Scrapeando links desde https://terraria.fandom.com/es/wiki/Gu%C3%ADa_del_principiante
	Links revisados: 439/439
Cantidad de urls: 100


In [ ]:
# Scrapeamos el texto de cada url (con conteo OK/saltados)
url_text = []
i = 0
ok = 0
skipped = 0

for url in url_data:
  print(f"\rLinks scrapeados: {i}/{len(url_data)}", end="")
  try:
      t = get_text(url)[0]
      if t:
          url_text.append(t)
          ok += 1
      else:
          skipped += 1
  except KeyboardInterrupt:
      raise
  except Exception:
      skipped += 1
  i += 1

print(f"\rLinks scrapeados: {i}/{len(url_data)}")
print(f"Leídos OK: {ok} | Saltados: {skipped} | Total: {len(url_data)}")

url_text


Links scrapeados: 100/100
Leídos OK: 100 | Saltados: 0 | Total: 100


['Esta página muestra las IDs de buffs utilizadas internamente en el código de Terraria para referirse tanto a los efectos positivos como a los negativos.\n\nSumario\n1\t1 - 70\n2\t71 - 140\n3\t141 - 210\n4\t211 - 280\n5\t281 - 326\n1 - 70\nExpandir\n71 - 140\nExpandir\n141 - 210\nExpandir\n211 - 280\nExpandir\n281 - 326\nExpandir\nExpandir\nIDs',
 'Esta página muestra las IDs de objetos utilizadas internamente en el código de Terraria para referirse a objetos. Esta información puede ser útil para propósitos de búsqueda, o en el desarrollo de contenido de terceros, como mods y mapas. No todos los objetos tienen una ID única; los objetos que comparten una ID están diferenciados por sus nombres.\n\nSumario\n1\tIDs por lanzamiento\n2\t1 - 300\n3\t301 - 600\n4\t601 - 900\n5\t901 - 1200\n6\t1201 - 1500\n7\t1501 - 1800\n8\t1801 - 2100\n9\t2101 - 2400\n10\t2401 - 2700\n11\t2701 - 3000\n12\t3001 - 3300\n13\t3301 - 3600\n14\t3601 - 3900\n15\t3901 - 4200\n16\t4201 - 4500\n17\t4501 - 4800\n18\t48

In [ ]:
print(i,ok,skipped)

100 100 0


In [ ]:
# Scrapeamos los links de salida de cada url
urls_out = []
ok = 0
skipped = 0
total = len(url_data)

for i, url in enumerate(url_data, 1):
    links_page = []
    print(f"({i}/{total})", end=" ")
    try:
        get_links(url, url_base, links_page)
        urls_out.append(links_page)
        ok += 1
    except KeyboardInterrupt:
        raise
    except Exception as e:
        print("\nSaltado por error:", e)
        urls_out.append([])
        skipped += 1

print(f"\nPáginas procesadas OK: {ok} | Páginas saltadas: {skipped} | Total: {total}")


(1/100) Scrapeando links desde https://terraria.fandom.com/es/wiki/Helado
	Links revisados: 150/150
(2/100) Scrapeando links desde https://terraria.fandom.com/es/wiki/Objetos_de_apariencia
	Links revisados: 529/529
(3/100) Scrapeando links desde https://terraria.fandom.com/es/wiki/Cuchillo_envenenado
	Links revisados: 579/579
(4/100) Scrapeando links desde https://terraria.fandom.com/es/wiki/Pesca
	Links revisados: 548/548
(5/100) Scrapeando links desde https://terraria.fandom.com/es/wiki/Mineral_de_platino
	Links revisados: 164/164
(6/100) Scrapeando links desde https://terraria.fandom.com/es/wiki/Mesa
	Links revisados: 269/269
(7/100) Scrapeando links desde https://terraria.fandom.com/es/wiki/Re-Logic
	Links revisados: 57/57
(8/100) Scrapeando links desde https://terraria.fandom.com/es/wiki/Modo_experto
	Links revisados: 481/481
(9/100) Scrapeando links desde https://terraria.fandom.com/es/wiki/Pir%C3%A1mide
	Links revisados: 77/77
(10/100) Scrapeando links desde https://terraria.fan

In [ ]:
# Construimos un dataframe y lo guardamos
df = pd.DataFrame(list(zip(url_data, url_text, urls_out)), columns=['url', 'text', 'urls_out'])
df.to_pickle('wiki_terraria.pkl')
df.head()

,url,text,urls_out
0,https://terraria.fandom.com/es/wiki/Helado,Esta página muestra las IDs de buffs utilizada...,[https://terraria.fandom.com/es/wiki/IDs_de_bu...
1,https://terraria.fandom.com/es/wiki/Objetos_de...,Esta página muestra las IDs de objetos utiliza...,"[https://terraria.fandom.com/es/wiki/Armadura,..."
2,https://terraria.fandom.com/es/wiki/Cuchillo_e...,"Para incluir un artículo, imagen o categoría a...",[https://terraria.fandom.com/es/wiki/Armas_arr...
3,https://terraria.fandom.com/es/wiki/Pesca,Las decoraciones son objetos que adornan cualq...,"[https://terraria.fandom.com/es/wiki/Agua, htt..."
4,https://terraria.fandom.com/es/wiki/Mineral_de...,Estación\n\n\nForja\n\nResultado\tIngredientes...,[https://terraria.fandom.com/es/wiki/Minerales...


## Tratamiento de datos

In [4]:
# Cargamos los datos
df = pd.read_pickle('wiki_terraria.pkl')
df.head()

,url,text,urls_out
0,https://terraria.fandom.com/es/wiki/Helado,Esta página muestra las IDs de buffs utilizada...,[https://terraria.fandom.com/es/wiki/IDs_de_bu...
1,https://terraria.fandom.com/es/wiki/Objetos_de...,Esta página muestra las IDs de objetos utiliza...,"[https://terraria.fandom.com/es/wiki/Armadura,..."
2,https://terraria.fandom.com/es/wiki/Cuchillo_e...,"Para incluir un artículo, imagen o categoría a...",[https://terraria.fandom.com/es/wiki/Armas_arr...
3,https://terraria.fandom.com/es/wiki/Pesca,Las decoraciones son objetos que adornan cualq...,"[https://terraria.fandom.com/es/wiki/Agua, htt..."
4,https://terraria.fandom.com/es/wiki/Mineral_de...,Estación\n\n\nForja\n\nResultado\tIngredientes...,[https://terraria.fandom.com/es/wiki/Minerales...


In [5]:
# Función text_treatment
# Recibe un texto de entrada, retornandolo limpio en forma de lemmas
def text_treatment(text):

  #Eliminar tags html
  html_tags = re.compile('<.*?>|&([a-z0-9]+|#[0-9]{1,6}|#x[0-9a-f]{1,6});')
  text = re.sub(html_tags, ' ', text)

  #Eliminar acentos
  text = strip_accents_ascii(text)

  #Pasar a minúscula
  text = text.lower()

  #Eliminar puntuación
  regex = re.compile('[%s]' % re.escape(string.punctuation))
  text = regex.sub('', text)

  #Eliminar espacios adicionales
  text = ' '.join(text.split())

  #Remover stopwords
  spanish_stopwords = stopwords.words('spanish')
  text = ' '.join(word for word in text.split() if word not in spanish_stopwords)

  #Lematizacion
  nlp = spacy.load('es_core_news_md')
  text = nlp(text)
  text = ' '.join(tok.lemma_.lower() for tok in text)

  return text

In [6]:
# Limpiamos los textos de cada url
df['text'] = df['text'].apply(lambda x: text_treatment(x))
df.head()

,url,text,urls_out
0,https://terraria.fandom.com/es/wiki/Helado,pagina mostrar id buff utilizado internamente ...,[https://terraria.fandom.com/es/wiki/IDs_de_bu...
1,https://terraria.fandom.com/es/wiki/Objetos_de...,pagina mostrar id objeto utilizado internament...,"[https://terraria.fandom.com/es/wiki/Armadura,..."
2,https://terraria.fandom.com/es/wiki/Cuchillo_e...,incluir articulo imagen categorio categoria an...,[https://terraria.fandom.com/es/wiki/Armas_arr...
3,https://terraria.fandom.com/es/wiki/Pesca,decoración objeto adornar cualquiera tipo cons...,"[https://terraria.fandom.com/es/wiki/Agua, htt..."
4,https://terraria.fandom.com/es/wiki/Mineral_de...,estacion forja resultado ingredient viga adama...,[https://terraria.fandom.com/es/wiki/Minerales...


# Ranking de páginas

## WordRank

In [31]:
# Función get_top_n
# retorna una lista de las n palabras mas relevantes para cada documento

def get_top_n(terms, X, n=150):
    tops = []
    for i in range(X.shape[0]):
        row = X.getrow(i)
        if row.nnz == 0:
            tops.append({})
            continue

        idx = row.indices         # índices de términos presentes en el doc i
        vals = row.data           # pesos TF-IDF correspondientes

        # Si hay más de n términos en el doc, se toma el topn sin ordenar todo
        if len(vals) > n:
            topk_idx = np.argpartition(vals, -n)[-n:]
            # se ordena desc en los n seleccionados
            order = topk_idx[np.argsort(vals[topk_idx])[::-1]]
        else:
            order = np.argsort(vals)[::-1]

        feats = idx[order]
        weights = vals[order]     # pesos TF-IDF ordenados DESC
        # dict termino ---> peso (incluye unigramas y bigramas si es que el vectorizados los gener´p)
        tops.append({terms[j]: float(w) for j, w in zip(feats, weights)})

    return tops


In [32]:
# - ngram_range=(1,2) (unigramas + bigramas).
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),   #unigramas y bigramas
    min_df=2,
    lowercase=False       # ya se hizo
)

# Matriz documento–término de TF-IDF
X = vectorizer.fit_transform(df['text'])

# Vocabulario alineado a X
terms = vectorizer.get_feature_names_out()

# Construcción de 'wordrank' por documento
df['wordrank'] = get_top_n(terms, X, n=50)


In [33]:
df.head()

,url,text,urls_out,wordrank,pr_085,pr_060,hits_hub,hits_auth
0,https://terraria.fandom.com/es/wiki/Helado,pagina mostrar id buff utilizado internamente ...,[https://terraria.fandom.com/es/wiki/IDs_de_bu...,"{'expandir': 0.34098133269736186, '326': 0.304...",0.002195,0.004688,0.046600,0.021261
1,https://terraria.fandom.com/es/wiki/Objetos_de...,pagina mostrar id objeto utilizado internament...,"[https://terraria.fandom.com/es/wiki/Armadura,...","{'objeto': 0.6487023868120015, 'expandir': 0.4...",0.006860,0.009278,0.018973,0.000541
2,https://terraria.fandom.com/es/wiki/Cuchillo_e...,incluir articulo imagen categorio categoria an...,[https://terraria.fandom.com/es/wiki/Armas_arr...,"{'categoria': 0.3733176310357018, 'categoria e...",0.005302,0.007573,0.246596,0.195639
3,https://terraria.fandom.com/es/wiki/Pesca,decoración objeto adornar cualquiera tipo cons...,"[https://terraria.fandom.com/es/wiki/Agua, htt...","{'decoración': 0.37026463498598833, 'forja': 0...",0.014343,0.012651,0.087781,0.053001
4,https://terraria.fandom.com/es/wiki/Mineral_de...,estacion forja resultado ingredient viga adama...,[https://terraria.fandom.com/es/wiki/Minerales...,"{'bloque piedra': 0.45373801251596185, 'minera...",0.018415,0.012590,0.068751,0.059215


## PageRank

In [34]:
pagerank = np.zeros((len(df), len(df)), dtype=np.float32)

# Mapa URL -> índice para acelerar la búsqueda de columnas
url_to_idx = {u: i for i, u in enumerate(df['url'])}

# Construimos la matriz de adyacencia
for i, row in df.iterrows():
  outs = row['urls_out'] or []
  for url in outs:
    j = url_to_idx.get(url, None)  # solo si la URL destino está en el corpus
    if j is not None:
      pagerank[i][j] = 1.0

# outdeg: grado de salida por fila (nodo)
outdeg = pagerank.sum(axis=1).astype(int)

# inlinks: lista de predecesores por columna (quiénes apuntan a j)
inlinks = [np.where(pagerank[:, j] != 0)[0].tolist() for j in range(len(df))]

pagerank


array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 1., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 1.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [35]:
# Implementación de PageRank
def pagerank_from_scratch(inlinks, outdeg, d=0.85, tol=1e-6, max_iter=100):
    """
    inlinks[i]: lista de nodos j con enlace j -> i
    outdeg[j]:  grado de salida de j
    entrega: vector PR (suma = 1).
    """
    n = len(inlinks)
    pr = np.full(n, 1.0 / n, dtype=np.float64)
    base = (1.0 - d) / n

    for _ in range(max_iter):
        dangling_mass = pr[outdeg == 0].sum()

        # nuevo PR parte en el término base + contrib. de dangling
        new_pr = np.full(n, base, dtype=np.float64)
        new_pr += d * (dangling_mass / n)

        # aportes entrantes
        for i in range(n):
            s = 0.0
            for j in inlinks[i]:
                s += pr[j] / outdeg[j]
            new_pr[i] += d * s

        # convergencia
        if np.abs(new_pr - pr).sum() < tol:
            pr = new_pr
            break
        pr = new_pr

    pr /= pr.sum()
    return pr

# d=0.85
pr_085 = pagerank_from_scratch(inlinks, outdeg, d=0.85, tol=1e-6, max_iter=100)
pr_060 = pagerank_from_scratch(inlinks, outdeg, d=0.60, tol=1e-6, max_iter=100)

df['pr_085']  = pr_085
df['pr_060'] = pr_060
df.head()


,url,text,urls_out,wordrank,pr_085,pr_060,hits_hub,hits_auth
0,https://terraria.fandom.com/es/wiki/Helado,pagina mostrar id buff utilizado internamente ...,[https://terraria.fandom.com/es/wiki/IDs_de_bu...,"{'expandir': 0.34098133269736186, '326': 0.304...",0.002195,0.004688,0.046600,0.021261
1,https://terraria.fandom.com/es/wiki/Objetos_de...,pagina mostrar id objeto utilizado internament...,"[https://terraria.fandom.com/es/wiki/Armadura,...","{'objeto': 0.6487023868120015, 'expandir': 0.4...",0.006860,0.009278,0.018973,0.000541
2,https://terraria.fandom.com/es/wiki/Cuchillo_e...,incluir articulo imagen categorio categoria an...,[https://terraria.fandom.com/es/wiki/Armas_arr...,"{'categoria': 0.3733176310357018, 'categoria e...",0.005302,0.007573,0.246596,0.195639
3,https://terraria.fandom.com/es/wiki/Pesca,decoración objeto adornar cualquiera tipo cons...,"[https://terraria.fandom.com/es/wiki/Agua, htt...","{'decoración': 0.37026463498598833, 'forja': 0...",0.014343,0.012651,0.087781,0.053001
4,https://terraria.fandom.com/es/wiki/Mineral_de...,estacion forja resultado ingredient viga adama...,[https://terraria.fandom.com/es/wiki/Minerales...,"{'bloque piedra': 0.45373801251596185, 'minera...",0.018415,0.012590,0.068751,0.059215


## HITS

In [36]:
# HITS

n = pagerank.shape[0]

# A quienes apunta i
outlinks_idx = [np.where(pagerank[i, :] != 0)[0].tolist() for i in range(n)]
# A quienes apunta j
inlinks_idx  = [np.where(pagerank[:, j] != 0)[0].tolist() for j in range(n)]

hubs = np.ones(n, dtype=np.float64)
auth = np.ones(n, dtype=np.float64)

# normalizacion
def _norm2(v):
    s = np.linalg.norm(v)
    return v / s if s > 0 else v

hubs = _norm2(hubs)
auth = _norm2(auth)
tol = 1e-6
max_iter = 200

for _ in range(max_iter):
    # Actualiza authorities
    new_auth = np.zeros(n, dtype=np.float64)
    for i in range(n):
        if inlinks_idx[i]:                        # si alguna pag apunta a i
            new_auth[i] = sum(hubs[j] for j in inlinks_idx[i])

    # Actualiza hubs a partir de nuevas authorities
    new_hubs = np.zeros(n, dtype=np.float64)
    for i in range(n):
        if outlinks_idx[i]:                       # si i apunta a otros
            new_hubs[i] = sum(new_auth[k] for k in outlinks_idx[i])

    new_auth = _norm2(new_auth)
    new_hubs = _norm2(new_hubs)

    # Criterio de convergencia, norma l1
    delta = np.abs(new_auth - auth).sum() + np.abs(new_hubs - hubs).sum()
    auth, hubs = new_auth, new_hubs
    if delta < tol:
        break

top_h = np.argsort(-hubs)[:10]
top_a = np.argsort(-auth)[:10]
print("Hubs (top 10):", [(int(i), float(hubs[i])) for i in top_h])
print("Authorities (top 10):", [(int(i), float(auth[i])) for i in top_a])


Hubs (top 10): [(98, 0.26931736964903424), (29, 0.2639542571082441), (58, 0.2529782060348599), (10, 0.24756639874615285), (2, 0.24659640700608043), (31, 0.2461659123190256), (41, 0.24289698609240237), (51, 0.23564189943426017), (13, 0.22919017975505895), (11, 0.20703538338469582)]
Authorities (top 10): [(66, 0.29235210973848563), (35, 0.2664127840089547), (36, 0.22096365221653624), (51, 0.21572787522550557), (67, 0.21572787522550557), (49, 0.21178710681017857), (41, 0.20884379128280403), (11, 0.208493887573048), (98, 0.20679791551911908), (19, 0.2062750125211031)]


In [37]:
# Almacenamos los valores en el dataframe
df['hits_hub']  = hubs
df['hits_auth'] = auth
df.head()

,url,text,urls_out,wordrank,pr_085,pr_060,hits_hub,hits_auth
0,https://terraria.fandom.com/es/wiki/Helado,pagina mostrar id buff utilizado internamente ...,[https://terraria.fandom.com/es/wiki/IDs_de_bu...,"{'expandir': 0.34098133269736186, '326': 0.304...",0.002195,0.004688,0.046600,0.021261
1,https://terraria.fandom.com/es/wiki/Objetos_de...,pagina mostrar id objeto utilizado internament...,"[https://terraria.fandom.com/es/wiki/Armadura,...","{'objeto': 0.6487023868120015, 'expandir': 0.4...",0.006860,0.009278,0.018973,0.000541
2,https://terraria.fandom.com/es/wiki/Cuchillo_e...,incluir articulo imagen categorio categoria an...,[https://terraria.fandom.com/es/wiki/Armas_arr...,"{'categoria': 0.3733176310357018, 'categoria e...",0.005302,0.007573,0.246596,0.195639
3,https://terraria.fandom.com/es/wiki/Pesca,decoración objeto adornar cualquiera tipo cons...,"[https://terraria.fandom.com/es/wiki/Agua, htt...","{'decoración': 0.37026463498598833, 'forja': 0...",0.014343,0.012651,0.087781,0.053001
4,https://terraria.fandom.com/es/wiki/Mineral_de...,estacion forja resultado ingredient viga adama...,[https://terraria.fandom.com/es/wiki/Minerales...,"{'bloque piedra': 0.45373801251596185, 'minera...",0.018415,0.012590,0.068751,0.059215


# Lógica de búsqueda

## Funciones de búsqueda

In [38]:
# Procesa la consulta y genera unigramas + bigramas, para asi matchear contra el wordrank que incluye n gramas (1,2)
def _query_terms(sentence):
  q = text_treatment(sentence)
  toks = nltk.word_tokenize(q)
  bigrams = [f"{toks[i]} {toks[i+1]}" for i in range(len(toks)-1)]
  return toks + bigrams

In [42]:
def _search_with_struct(sentence, df, struct_col, alpha=0.5):
  q_terms = _query_terms(sentence)
  results = {}
  for _, row in df.iterrows():
    wr = row['wordrank'] or {}
    p_wordrank = 0.0
    for t in q_terms:
      if t in wr:
        p_wordrank += wr[t]
    if p_wordrank != 0:
      struct = row.get(struct_col, None)
      if struct is None:
        continue
      score = alpha * float(struct) + (1 - alpha) * float(p_wordrank)
      results[row['url']] = score

  #Si no hay match semántico, entonces un fallback estructural
  if not results:
    tmp = df[['url', struct_col]].dropna()
    tmp = tmp.sort_values(struct_col, ascending=False).head(10)
    return [(u, float(s)) for u, s in zip(tmp['url'], tmp[struct_col])]

  sorted_list = sorted(results.items(), key=lambda x: x[1], reverse=True)
  return sorted_list[:10]


In [44]:
# Pagerank con d=0.85
# Busqueda usando pagerank con d=0.85 y wordrank
def search_pagerank(sentence, df):
  return _search_with_struct(sentence, df, struct_col='pr_085', alpha=0.5)

# Pagerank con d 0.60
def search_pagerank_alt(sentence, df):
  col = 'pr_060'
  return _search_with_struct(sentence, df, struct_col=col, alpha=0.5)

# Usando authorities como señal principal
def search_hits(sentence, df):
  return _search_with_struct(sentence, df, struct_col='hits_auth', alpha=0.5)

#SALSA
def search_salsa(sentence, df):
  return _search_with_struct(sentence, df, struct_col='salsa_auth', alpha=0.5)


In [54]:
top_terraria = search_salsa("Como matar al muro de carne en modo experto", df)
for i, (url, score) in enumerate(top_terraria, start=1):
  print(f"{i:>2}) {score:.4f}  {url}")

## Consultas

# Conclusiones

Ingrese aquí sus conclusiones.